In [1]:
import pandas as pd

melb_df = pd.read_csv('data/melb_data_fe.csv')

Одна из основных задач анализа данных — группировка данных и сравнение показателей в группах. Например, необходимость сравнить средний уровень заработной платы в зависимости от пола/уровня образования/региона. Или же проследить, какая группа клиентов приносит наибольший доход, чтобы направить свое внимание на эту группу.

В некоторых случаях группировки может быть достаточно, чтобы ответить на вопросы бизнеса. В других случаях это может стать первым шагом в более сложном анализе. Так, например, благодаря группировке мы можем выявлять признаки, которые не несут никакой статистической значимости, или признаки, которые вносят наибольший вклад.

Так или иначе, владение группировкой — важный навык, который открывает новые возможности по работе с данными.

В библиотеке Pandas для группировки данных по одному или нескольким признакам можно использовать метод groupby().

Основные параметры метода groupby():

- by — имя или список имен столбцов, по которым производится группировка;
- axis — ось, по которой производится группировка (0 — строка, 1 — столбец). По умолчанию 0 (группировка производится по строкам);
- as_index — добавляется ли дополнительный индекс к таблице. По умолчанию — True.

Метод groupby() возвращает объект DataFrameGroupBy, который хранит в себе информацию о том, какие строки относятся к определенной группе, и сам по себе не представляет для нас интереса.

Однако к этому объекту можно применить агрегирующие методы (mean, median, sum и тд), чтобы рассчитывать показатели внутри каждой группы.

![Схема работы метода groupby()](data/groupby_scheme.png)

Сначала мы разделяем данные на группы с помощью метода groupby(), после чего к каждой группе применяем агрегацию и объединяем результаты в новую таблицу.

Применим агрегирующую функцию среднего к результату работы groupby(). В качестве столбца группировки возьмем столбец типа объекта недвижимости (Type):

In [2]:
melb_df.groupby(by='Type').mean(numeric_only=True)

,Rooms,Price,Distance,Postcode,Bedroom,Bathroom,Car,Landsize,BuildingArea,Lattitude,Longtitude,Propertycount,MeanRoomsSquare,AreaRatio,MonthSale,AgeBuilding,WeekdaySale,Weekend
Type,,,,,,,,,,,,,,,,,,
house,3.260874,1.242665e+06,10.979479,3104.080643,3.229336,1.613822,1.772674,617.181924,152.162553,-37.803795,144.994700,7259.025505,18.996731,-0.490031,6.750873,55.669700,4.015769,0.689808
townhouse,2.837522,9.337351e+05,9.851346,3100.777379,2.814183,1.809695,1.555655,279.606822,134.649710,-37.815782,144.996489,7094.459605,18.569847,-0.094916,6.621185,26.690305,3.980251,0.681329
unit,1.963871,6.051275e+05,7.607391,3110.797481,1.966523,1.183295,1.128936,477.314219,102.235863,-37.823710,144.996363,8199.280080,21.068242,0.319883,6.578721,39.703016,3.817368,0.646006


Получили таблицу, на пересечении строк и столбцов которой находятся средние значения каждого числового признака в наших данных.

Обращаем внимание на структуру получившейся таблицы: теперь на месте индексов стоят значения типа объекта недвижимости Type (house, townhouse, unit).

Если нужно видеть тип объекта в качестве отдельного столбца, можно выставить параметр as_index на False.

Как правило, нам не нужна информация обо всех столбцах, поэтому агрегирующие методы можно применять только к интересующему нас столбцу. Сравним средние цены на объекты в зависимости от их типа:

In [3]:
melb_df.groupby(by='Type')['Price'].mean()

Type
house        1.242665e+06
townhouse    9.337351e+05
unit         6.051275e+05
Name: Price, dtype: float64

Так как мы считаем только один показатель (среднее) для одного столбца, в результате получаем объект Series.

Из таблицы видно, что наибольшей средней ценой обладают объекты типа house (дома, коттеджи, виллы). Можно сделать вывод, что тип постройки является значимым фактором при определении цены объекта недвижимости.

Теперь выясним, какие регионы наиболее удалены от центра Мельбурна. Для этого найдем минимальное значение расстояния от центра города до объекта в зависимости от его региона. Результат отсортируем по убыванию расстояния:

In [ ]:
# найдем минимальное значение расстояния от центра города до объекта в зависимости от его региона. 
# Результат отсортируем по убыванию расстояния
melb_df.groupby(by='Regionname')['Distance'].min().sort_values(ascending=False)

Regionname
Western Victoria              29.8
Eastern Victoria              25.2
Northern Victoria             21.8
South-Eastern Metropolitan    14.7
Eastern Metropolitan           7.8
Western Metropolitan           4.3
Southern Metropolitan          0.7
Northern Metropolitan          0.0
Name: Distance, dtype: float64

Наиболее удаленными являются все районы Victoria.

Чтобы рассчитать несколько агрегирующих методов, можно воспользоваться методом agg(), который принимает список строк с названиями агрегаций.

Построим таблицу для анализа продаж по месяцам. Найдем количество продаж, а также среднее и максимальное значения цен объектов недвижимости (Price), сгруппированных по номеру месяца продажи (MonthSale). Результат отсортируем по количеству продаж в порядке убывания:

In [ ]:
# Найдем количество продаж, а также среднее и максимальное значения цен объектов недвижимости (Price), 
# сгруппированных по номеру месяца продажи (MonthSale). Результат отсортируем по количеству продаж в 
# порядке убывания
melb_df.groupby('MonthSale')['Price'].agg(
    ['count', 'mean', 'max']
).sort_values(by='count', ascending=False)

,count,mean,max
MonthSale,,,
8,1850,1.056371e+06,6500000.0
7,1835,9.314698e+05,9000000.0
5,1644,1.097807e+06,8000000.0
6,1469,1.068981e+06,7650000.0
3,1408,1.146762e+06,5600000.0
4,1246,1.050479e+06,5500000.0
9,1188,1.126349e+06,6400000.0
10,854,1.135970e+06,6250000.0
11,750,1.142503e+06,5050000.0


Так как мы считаем несколько показателей для одного столбца, в результате мы получаем датафрейм.

В результате применения метода agg(), в который мы передали список с названиями интересующих нас агрегирующих функций, мы получаем датафрейм со столбцами count, mean, max, где для каждого месяца рассчитаны соответствующие параметры. Результат сортируем по столбцу count.

Какие выводы можно сделать из этой таблицы:

- Пик продаж приходится на период весна-лето;
- Средняя цена продаваемых объектов относительно стабильна и находится в пределах 1 млн. австралийских долларов с небольшими отклонениями (около 100 тыс. влево и вправо);
- Прослеживается некоторая зависимость между сезоном и максимальной ценой объектов: в месяцы с большим спросом на объекты недвижимости цена также имеет наибольшие показатели. Можно сделать предположение, что это связано с повышением цен на элитные дома в периоды большого спроса.

Если нужна полная информация обо всех основных статистических характеристиках внутри каждой группы, можно воспользоваться методом agg(), передав в качестве его параметра строку 'describe':

In [6]:
melb_df.groupby('MonthSale')['Price'].agg('describe')

,count,mean,std,min,25%,50%,75%,max
MonthSale,,,,,,,,
1,278.0,9.397921e+05,577668.924214,170000.0,570500.0,795000.0,1111250.0,5200000.0
2,333.0,1.169051e+06,671564.357417,131000.0,710000.0,1020000.0,1478000.0,4735000.0
3,1408.0,1.146762e+06,709573.596867,85000.0,680000.0,945000.0,1400000.0,5600000.0
4,1246.0,1.050479e+06,591892.902979,145000.0,655000.0,905500.0,1298750.0,5500000.0
5,1644.0,1.097807e+06,668492.867996,145000.0,650000.0,905000.0,1371250.0,8000000.0
6,1469.0,1.068981e+06,606010.069052,222000.0,660000.0,900000.0,1325000.0,7650000.0
7,1835.0,9.314698e+05,537390.803161,190000.0,586750.0,800000.0,1150000.0,9000000.0
8,1850.0,1.056371e+06,619617.476541,160000.0,635000.0,892000.0,1310000.0,6500000.0
9,1188.0,1.126349e+06,608734.690742,170000.0,725000.0,980000.0,1360000.0,6400000.0


После базовых математических функций наиболее частым агрегированием является подсчет числа уникальных значений. Например, мы можем вычислить число уникальных риелторских компаний в зависимости от региона, чтобы понять, в каких регионах конкуренция на рынке недвижимости меньше. Это можно сделать, передав в параметр agg() строку 'nunique'.

Метод agg() поддерживает использование и других функций. Например, передадим дополнительно встроенную функцию set, чтобы получить множество из агентов недвижимости, которые работают в каждом из регионов:

In [ ]:
melb_df.groupby('Regionname')['SellerG'].agg(
    ['nunique', set]
).sort_values('nunique', ascending=False) # дополнительно сам отсортировал по убыванию

,nunique,set
Regionname,,
Northern Metropolitan,40,"{Nick, RT, Barry, Miles, Biggin, Kay, Eview, o..."
Southern Metropolitan,38,"{Nick, RT, Barry, Biggin, Kay, Eview, other, G..."
Western Metropolitan,34,"{Douglas, RT, Barry, Biggin, Greg, other, HAR,..."
Eastern Metropolitan,26,"{RT, Barry, Miles, Biggin, Kay, other, HAR, No..."
South-Eastern Metropolitan,25,"{Barry, Biggin, Eview, Greg, other, HAR, Noel,..."
Eastern Victoria,11,"{Fletchers, Barry, McGrath, Ray, hockingstuart..."
Northern Victoria,11,"{LITTLE, Barry, McGrath, Raine, Ray, hockingst..."
Western Victoria,6,"{Raine, Ray, hockingstuart, other, YPA, HAR}"


Как и ожидалось, наименьшая конкуренция в наиболее удаленном регионе Western Victoria, а наибольшая — в центральном районе Northern Metropolitain.

In [20]:
melb_df.groupby('Rooms')['Price'].mean()

Rooms
1     4.338245e+05
2     7.750812e+05
3     1.076081e+06
4     1.445282e+06
5     1.870260e+06
6     1.849366e+06
7     1.920700e+06
8     1.602750e+06
10    9.000000e+05
Name: Price, dtype: float64

In [29]:
melb_df.groupby('Regionname')['Lattitude'].std().sort_values(ascending=False)

Regionname
Eastern Victoria              0.147067
Northern Victoria             0.084455
South-Eastern Metropolitan    0.073411
Western Metropolitan          0.051251
Northern Metropolitan         0.049639
Eastern Metropolitan          0.047890
Southern Metropolitan         0.043080
Western Victoria              0.011579
Name: Lattitude, dtype: float64

In [30]:
date1 = pd.to_datetime('2017-05-01')
date2 = pd.to_datetime('2017-09-01')
mask = (date1 <= melb_df['Date']) & (melb_df['Date'] <= date2)
melb_df[mask].groupby('SellerG')['Price'].sum().sort_values(ascending=True)

SellerG
LITTLE             2742000.0
Cayzer             4439000.0
Burnham            4550500.0
Moonee             7328000.0
Thomson            8332000.0
Bells              8656000.0
Alexkarbon        10985000.0
McDonald          14637500.0
Rendina           15422276.0
Nick              16890000.0
Douglas           18341000.0
Buckingham        19033000.0
C21               19515000.0
Eview             19791500.0
Collins           20217000.0
Philip            22051800.0
Chisholm          23225000.0
Williams          23297000.0
Love              23365500.0
Purplebricks      23401000.0
O'Brien           23855508.0
HAR               25568000.0
Village           26473000.0
RW                29261000.0
Raine             30687700.0
Stockdale         35409800.0
Sweeney           36882750.0
Gary              39138400.0
Hodges            43231000.0
YPA               46354350.0
Miles             47582000.0
Kay               48569500.0
RT                50498000.0
Brad              55955000.0
Jas   